# Qwen3-0.6b G検定 Evaluation — Letter scoring

このNotebookはOpenAI評価と同じ `MMLUEvaluator` を使います。
通常は次の **User settings** だけを変更し、GPU runtimeでRun allしてください。

処理順: Settings → Environment setup → Model loading → Dataset / evaluator setup
→ Preflight / manifest → Evaluation → Results


## 1. Settings

### User settings — 通常はこのセルだけを変更

In [ ]:
from pathlib import Path
import json

PROJECT_DIR = Path("/content/AIkenSGTv1_New")
DATA_ROOT = Path("/content/gkentei-reference")
DATA_DIR = DATA_ROOT / "data"

# Evaluation repository
if not (PROJECT_DIR / "mmlu_eval").is_dir():
    !rm -rf /content/AIkenSGTv1_New
    !git clone https://github.com/HayatoHongo/AIkenSGTv1.git /content/AIkenSGTv1_New
    !cd /content/AIkenSGTv1_New && git switch tayama

# G検定データ: eval1=test, eval2=dev
from urllib.request import urlretrieve
import csv

DATA_ROOT.mkdir(parents=True, exist_ok=True)
(DATA_DIR / "dev").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "test").mkdir(parents=True, exist_ok=True)

DATA_URLS = {
    "test": "https://raw.githubusercontent.com/HayatoHongo/AIkenSGTv1/main/gkentei_eval1.jsonl",
    "dev": "https://raw.githubusercontent.com/HayatoHongo/AIkenSGTv1/main/gkentei_eval2.jsonl",
}

for split, url in DATA_URLS.items():
    jsonl_path = DATA_ROOT / f"gkentei_{split}.jsonl"
    csv_path = DATA_DIR / split / f"gkentei_{split}.csv"
    if not jsonl_path.exists():
        urlretrieve(url, jsonl_path)
    with jsonl_path.open(encoding="utf-8") as source, csv_path.open("w", newline="", encoding="utf-8") as target:
        writer = csv.writer(target)
        for line in source:
            item = json.loads(line)
            correct = chr(ord("A") + int(item["correct_answers"]) - 1)
            writer.writerow([item["question_text"], item["option_1"], item["option_2"], item["option_3"], item["option_4"], correct])

OUTPUT_DIR = Path("/content/drive/MyDrive/qwen3_gkentei_results")

SUBJECT = "gkentei"
NTRAIN = 3
SAMPLE_FRAC = 1.0
SEED = 42
LIMIT = 0
MANIFEST_PATH = None

### Project settings — 通常の問題セット変更では編集不要

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"
TOKENIZER = MODEL_ID
MODEL_FORMAT = "hf"

DEVICE = "cuda"
DTYPE = "bfloat16"
BATCH_SIZE = 1
MAX_CONTEXT_LENGTH = 2048

CONTEXT_POLICY = "reduce"
CONTEXT_TOKENIZER = "gpt2"

PERMUTATION_COUNT = 4
PERMUTATION_SEED = 0

SCORING_METHOD = "letter"
OUTPUT_NAME = f"qwen3_0.6b_gkentei_{NTRAIN}shot_letter.csv"

## 2. Environment setup

In [ ]:
%pip install -q numpy pandas tiktoken transformers safetensors huggingface_hub


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, str(PROJECT_DIR))

assert (PROJECT_DIR / "mmlu_eval").is_dir(), (
    "PROJECT_DIR must point to the repository containing mmlu_eval/"
)
assert DATA_DIR.is_dir(), "DATA_DIR must contain dev/ and test/"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Model loading

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from mmlu_eval.backends.aikengpt_backend import (
    AIkenGPTBackend,
)

assert DEVICE != "cuda" or torch.cuda.is_available(), "Select a Colab GPU runtime"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=getattr(torch, DTYPE),
).to(DEVICE)
model.config.use_cache = False
print("Loaded:", MODEL_ID)


## 4. Dataset / evaluator setup

In [ ]:
from mmlu_eval import EvalConfig, MMLUEvaluator

eval_config = EvalConfig(
    ntrain=NTRAIN,
    sample_frac=SAMPLE_FRAC,
    seed=SEED,
    limit=LIMIT,
    subject=SUBJECT,
    permutation_count=PERMUTATION_COUNT,
    permutation_seed=PERMUTATION_SEED,
    context_policy=CONTEXT_POLICY,
    context_tokenizer=CONTEXT_TOKENIZER,
    max_context_length=MAX_CONTEXT_LENGTH,
)
evaluator = MMLUEvaluator(DATA_DIR, eval_config)


## 5. Preflight / manifest

In [ ]:
from mmlu_eval.core import atomic_csv

# Existing manifest is authoritative. Otherwise create one once and reuse it below.
manifest = evaluator.manifest(MANIFEST_PATH)
preflight_manifest_path = OUTPUT_DIR / (Path(OUTPUT_NAME).stem + ".preflight.manifest.csv")
preflight_prompts_path = OUTPUT_DIR / (Path(OUTPUT_NAME).stem + ".preflight.prompts.jsonl")
atomic_csv(preflight_manifest_path, manifest)
evaluator.export_debug(manifest, preflight_prompts_path)
active_manifest_path = Path(MANIFEST_PATH) if MANIFEST_PATH is not None else preflight_manifest_path

first = manifest.iloc[0]
first_case = evaluator.cases(first.subject, int(first.test_index))[0]
print(f"Questions: {len(manifest)} / prompts: {len(manifest) * 4}")
print("Manifest:", active_manifest_path)
print("\nFirst prompt:\n")
print(first_case.prompt)


## 6. Evaluation

In [ ]:
backend = AIkenGPTBackend(
    model,
    tokenizer,
    model_identifier=MODEL_ID,
    tokenizer_identifier=TOKENIZER,
    scoring_method=SCORING_METHOD,
    text_reduction=None,
    model_format=MODEL_FORMAT,
    device=DEVICE,
    dtype=DTYPE,
    batch_size=BATCH_SIZE,
    max_context_length=MAX_CONTEXT_LENGTH,
    checkpoint_sha256=None,
)

OUTPUT_PATH = OUTPUT_DIR / OUTPUT_NAME
results = evaluator.run(
    backend,
    OUTPUT_PATH,
    manifest_path=active_manifest_path,
)


## 7. Results

In [ ]:
from mmlu_eval.core import summarize

print(summarize(results))
print("Result CSV:", OUTPUT_PATH)
print("Manifest:", OUTPUT_PATH.with_suffix(".manifest.csv"))
print("Prompts:", OUTPUT_PATH.with_suffix(".prompts.jsonl"))
display(results.head())

# OpenAI runとの入力一致を確認する場合:
# from mmlu_eval.compare import compare
# compare("/path/to/openai.prompts.jsonl", OUTPUT_PATH.with_suffix(".prompts.jsonl"))
